# LeetCode 1252 — Cells with Odd Values in a Matrix bằng Python

**Link đề:** https://leetcode.com/problems/cells-with-odd-values-in-a-matrix/
**Độ khó:** Easy · **Chủ đề:** Array, Math, Simulation

```python
class Solution:
    def oddCells(self, m: int, n: int, indices: list[list[int]]) -> int:
        ...
```

## Mục tiêu học
1. Biết tạo và duyệt **ma trận (list 2 chiều)** trong Python, tránh cạm bẫy `[[0] * n] * m`.
2. Viết lời giải **mô phỏng (simulation)** đúng theo đề — cách người mới nghĩ ra đầu tiên.
3. Nhận ra **nút thắt**: ta không cần biết giá trị từng ô, chỉ cần **tính chẵn/lẻ**.
4. Tách bài toán thành **đếm theo hàng + đếm theo cột**, rồi đi tới công thức đếm `O(m + n + L)` (đúng follow-up của đề).
5. Trace từng bước, chứng minh vì sao đúng, test đối chiếu 4 cách, lỗi hay gặp, biến thể và bài tập có đáp án.

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài và ví dụ

Có một ma trận `m x n` ban đầu toàn số `0`. Cho mảng `indices`, mỗi phần tử `indices[i] = [ri, ci]` là một vị trí (đánh số từ 0).
Với **mỗi** vị trí `[ri, ci]`, làm **cả hai** việc:

- Cộng 1 vào **mọi ô của hàng** `ri`.
- Cộng 1 vào **mọi ô của cột** `ci`.

Sau khi xử lý hết `indices`, trả về **số ô có giá trị lẻ**.

| `m` | `n` | `indices` | Output |
|---:|---:|---|---:|
| 2 | 3 | `[[0,1],[1,1]]` | `6` |
| 2 | 2 | `[[1,1],[0,0]]` | `0` |

**Constraints:**
- `1 <= m, n <= 50`
- `1 <= indices.length <= 100`
- `0 <= ri < m`, `0 <= ci < n`

**Follow up:** giải trong `O(n + m + indices.length)` thời gian với chỉ `O(n + m)` bộ nhớ phụ.

Chú ý: ô giao `(ri, ci)` bị cộng **2 lần** (một lần vì hàng, một lần vì cột).

### Tính tay ví dụ 1: `m = 2, n = 3, indices = [[0,1],[1,1]]`

```text
Ban đầu:          Sau [0,1]:              Sau [1,1]:
0 0 0             hàng 0 +1 -> 1 1 1      hàng 1 +1 -> 1 2 1
0 0 0             cột 1  +1 -> 1 2 1                   1 1 1
                               0 1 0      cột 1  +1 -> 1 3 1
                                                       1 3 1
```

Ma trận cuối `[[1,3,1],[1,3,1]]`: cả 6 ô đều lẻ → đáp án `6`.

Ví dụ 2: `[[1,1],[0,0]]` → mỗi hàng được cộng 1 lần, mỗi cột được cộng 1 lần → mọi ô = 2 → `0` ô lẻ.

In [1]:
# Kiểm tra lại bảng tay bằng cách in ma trận cuối (viết tay, chưa cần thuật toán)
final_1 = [[1, 3, 1], [1, 3, 1]]
final_2 = [[2, 2], [2, 2]]
print(sum(1 for row in final_1 for v in row if v % 2 == 1))  # 6
print(sum(1 for row in final_2 for v in row if v % 2 == 1))  # 0

6
0


## 2. Nền tảng Python cần biết

### 2.1. Ma trận = list chứa các list

Một ma trận `m x n` trong Python thường biểu diễn bằng list gồm `m` hàng, mỗi hàng là list `n` phần tử.
`grid[r][c]` đọc là "hàng `r`, cột `c`". Cách tạo an toàn là **list comprehension**:

```python
grid = [[0] * n for _ in range(m)]
```

- `[0] * n` tạo một list `n` số 0.
- `for _ in range(m)` lặp `m` lần; `_` là tên biến "không dùng tới".
- Mỗi lần lặp tạo **một list mới** → các hàng độc lập với nhau.

In [2]:
m, n = 2, 3
grid = [[0] * n for _ in range(m)]
print(grid)            # [[0, 0, 0], [0, 0, 0]]
grid[0][1] = 5
print(grid)            # [[0, 5, 0], [0, 0, 0]]  -> chỉ hàng 0 đổi
print(len(grid), len(grid[0]))   # 2 3  (số hàng, số cột)

[[0, 0, 0], [0, 0, 0]]
[[0, 5, 0], [0, 0, 0]]
2 3


### 2.2. Cạm bẫy kinh điển: `[[0] * n] * m`

Phép `* m` trên list **không sao chép** list con, mà lặp lại **cùng một tham chiếu** `m` lần.
Sửa một hàng là sửa tất cả các hàng! Đây là lỗi người mới gặp rất nhiều ở các bài ma trận.

In [3]:
bad = [[0] * 3] * 2
bad[0][1] = 5
print(bad)                    # [[0, 5, 0], [0, 5, 0]]  <- cả 2 hàng đều đổi!
print(bad[0] is bad[1])       # True: hai "hàng" là cùng một object

good = [[0] * 3 for _ in range(2)]
good[0][1] = 5
print(good)                   # [[0, 5, 0], [0, 0, 0]]
print(good[0] is good[1])     # False

[[0, 5, 0], [0, 5, 0]]
True
[[0, 5, 0], [0, 0, 0]]
False


### 2.3. Cộng 1 cho cả một hàng / cả một cột

- Cả hàng `r`: duyệt mọi cột `c` trong `range(n)` và làm `grid[r][c] += 1`.
- Cả cột `c`: duyệt mọi hàng `r` trong `range(m)` và làm `grid[r][c] += 1`.

`x += 1` là viết tắt của `x = x + 1`.

In [4]:
m, n = 2, 3
grid = [[0] * n for _ in range(m)]
r, c = 0, 1
for col in range(n):      # cả hàng 0
    grid[r][col] += 1
print(grid)               # [[1, 1, 1], [0, 0, 0]]
for row in range(m):      # cả cột 1
    grid[row][c] += 1
print(grid)               # [[1, 2, 1], [0, 1, 0]]  -> khớp bước 1 của ví dụ

[[1, 1, 1], [0, 0, 0]]
[[1, 2, 1], [0, 1, 0]]


### 2.4. Kiểm tra lẻ bằng `% 2` và đếm bằng `sum(...)`

- `v % 2` là **phần dư** khi chia cho 2: số lẻ cho `1`, số chẵn cho `0`.
- `sum(1 for ... if ...)` cộng 1 cho mỗi phần tử thỏa điều kiện → chính là **đếm**.
- Hai vòng `for` trong một biểu thức: `for row in grid for v in row` đọc như hai vòng lồng nhau (ngoài trước, trong sau).

In [5]:
print(7 % 2, 4 % 2, 0 % 2)    # 1 0 0
grid = [[1, 2, 1], [0, 1, 0]]
odd = sum(1 for row in grid for v in row if v % 2 == 1)
print(odd)                    # 3  (ba số 1)

# Viết dài tương đương:
cnt = 0
for row in grid:
    for v in row:
        if v % 2 == 1:
            cnt += 1
print(cnt)                    # 3

1 0 0
3
3


### 2.5. Tính chẵn lẻ của tổng (parity)

Quy tắc sẽ dùng rất nhiều ở phần sau:

| a | b | a + b |
|---|---|---|
| chẵn | chẵn | chẵn |
| chẵn | lẻ | **lẻ** |
| lẻ | chẵn | **lẻ** |
| lẻ | lẻ | chẵn |

Tức là `a + b` lẻ **khi và chỉ khi đúng một** trong hai số lẻ. Đây cũng chính là phép **XOR** (`^`) trên bit chẵn/lẻ: `(a % 2) ^ (b % 2)`.

In [6]:
for a in range(2):
    for b_ in range(2):
        print(f"a%2={a}, b%2={b_} -> (a+b)%2={(a + b_) % 2}, a^b={a ^ b_}")
# hai cột cuối luôn bằng nhau

a%2=0, b%2=0 -> (a+b)%2=0, a^b=0
a%2=0, b%2=1 -> (a+b)%2=1, a^b=1
a%2=1, b%2=0 -> (a+b)%2=1, a^b=1
a%2=1, b%2=1 -> (a+b)%2=0, a^b=0


## 3. Cách 1 — Mô phỏng đúng theo đề (Simulation)

Người mới thường nghĩ ngay: "đề bảo làm gì thì làm đúng như vậy".

1. Tạo ma trận `m x n` toàn 0.
2. Với mỗi `[r, c]`: cộng 1 cả hàng `r`, rồi cộng 1 cả cột `c`.
3. Đếm số ô lẻ.

Với constraints nhỏ (`m, n <= 50`, `L <= 100`), cách này hoàn toàn đủ nhanh để AC.

In [7]:
def odd_cells_simulate(m: int, n: int, indices: list[list[int]]) -> int:
    grid = [[0] * n for _ in range(m)]
    for r, c in indices:              # tách cặp [r, c] thành 2 biến
        for col in range(n):
            grid[r][col] += 1
        for row in range(m):
            grid[row][c] += 1
    return sum(1 for row in grid for v in row if v % 2 == 1)

print(odd_cells_simulate(2, 3, [[0, 1], [1, 1]]))   # 6
print(odd_cells_simulate(2, 2, [[1, 1], [0, 0]]))   # 0

6
0


### Giải thích `for r, c in indices`

Mỗi phần tử của `indices` là một list 2 phần tử, ví dụ `[0, 1]`. Viết `for r, c in indices` thì Python **tự tách** (unpacking): `r = 0`, `c = 1`.

In [8]:
for r, c in [[0, 1], [1, 1]]:
    print("r =", r, "| c =", c)
pair = [3, 7]
x, y = pair
print(x, y)   # 3 7

r = 0 | c = 1
r = 1 | c = 1
3 7


### Code nộp LeetCode — Cách 1

In [9]:
class Solution:
    def oddCells(self, m: int, n: int, indices: list[list[int]]) -> int:
        grid = [[0] * n for _ in range(m)]
        for r, c in indices:
            for col in range(n):
                grid[r][col] += 1
            for row in range(m):
                grid[row][c] += 1
        return sum(1 for row in grid for v in row if v % 2 == 1)

print(Solution().oddCells(2, 3, [[0, 1], [1, 1]]))  # 6

6


**Độ phức tạp Cách 1** (gọi `L = len(indices)`):

- Thời gian: mỗi thao tác tốn `n + m` bước → `O(L·(m + n))`, cộng thêm `O(m·n)` để đếm. Tổng `O(L·(m + n) + m·n)`.
- Bộ nhớ: `O(m·n)` cho ma trận.

## 4. Nút thắt — ta đang làm thừa việc gì?

- Ta cập nhật **từng ô** của cả hàng/cột, dù cuối cùng chỉ cần biết **ô nào lẻ**.
- Nếu `m, n` lên tới `10^5` thì ma trận `m·n` không thể tạo được, và mỗi thao tác tốn `m + n` bước.
- Câu hỏi then chốt: **giá trị cuối của ô `(r, c)` phụ thuộc vào cái gì?**

## 5. Insight — giá trị một ô = (số lần hàng được cộng) + (số lần cột được cộng)

Ô `(r, c)` chỉ được cộng 1 khi:
- có thao tác chọn **hàng** `r` (bất kể cột nào), hoặc
- có thao tác chọn **cột** `c` (bất kể hàng nào).

Nên nếu gọi `rows[r]` = số lần hàng `r` xuất hiện trong `indices`, `cols[c]` = số lần cột `c` xuất hiện, thì:

```text
grid[r][c] = rows[r] + cols[c]
```

Ví dụ 1: `indices = [[0,1],[1,1]]` → `rows = [1, 1]`, `cols = [0, 2, 0]`.
- `grid[0][0] = 1 + 0 = 1`, `grid[0][1] = 1 + 2 = 3`, `grid[1][2] = 1 + 0 = 1` … khớp `[[1,3,1],[1,3,1]]`.

In [10]:
indices = [[0, 1], [1, 1]]
m, n = 2, 3
rows = [0] * m
cols = [0] * n
for r, c in indices:
    rows[r] += 1
    cols[c] += 1
print("rows =", rows)    # [1, 1]
print("cols =", cols)    # [0, 2, 0]
rebuilt = [[rows[r] + cols[c] for c in range(n)] for r in range(m)]
print(rebuilt)           # [[1, 3, 1], [1, 3, 1]]

rows = [1, 1]
cols = [0, 2, 0]
[[1, 3, 1], [1, 3, 1]]


## 6. Cách 2 — Đếm hàng/cột rồi duyệt mọi ô

Từ insight: không cần cập nhật ma trận, chỉ cần 2 mảng đếm. Sau đó duyệt `m·n` ô và kiểm tra `(rows[r] + cols[c]) % 2 == 1`.

In [11]:
def odd_cells_count_rc(m: int, n: int, indices: list[list[int]]) -> int:
    rows = [0] * m
    cols = [0] * n
    for r, c in indices:
        rows[r] += 1
        cols[c] += 1
    odd = 0
    for r in range(m):
        for c in range(n):
            if (rows[r] + cols[c]) % 2 == 1:
                odd += 1
    return odd

print(odd_cells_count_rc(2, 3, [[0, 1], [1, 1]]))   # 6
print(odd_cells_count_rc(2, 2, [[1, 1], [0, 0]]))   # 0

6
0


**Độ phức tạp Cách 2:** thời gian `O(L + m·n)`, bộ nhớ `O(m + n)`. Đã bỏ được ma trận, nhưng vẫn duyệt `m·n` ô → chưa đạt follow-up.

## 7. Cách 3 — Công thức đếm `O(L + m + n)` (đáp án follow-up)

Dùng bảng parity ở mục 2.5: `rows[r] + cols[c]` lẻ ⇔ **đúng một** trong hai lẻ.

Gọi:
- `odd_r` = số hàng có `rows[r]` lẻ → còn `m - odd_r` hàng chẵn.
- `odd_c` = số cột có `cols[c]` lẻ → còn `n - odd_c` cột chẵn.

Ô lẻ gồm hai loại tách biệt:
- hàng lẻ × cột chẵn: `odd_r * (n - odd_c)` ô,
- hàng chẵn × cột lẻ: `(m - odd_r) * odd_c` ô.

```text
answer = odd_r * (n - odd_c) + (m - odd_r) * odd_c
```

Đây là **nguyên lý nhân** trong tổ hợp: chọn 1 hàng trong nhóm A và 1 cột trong nhóm B → `|A| · |B|` cách.

In [12]:
# Ví dụ 1: rows = [1, 1], cols = [0, 2, 0]
rows, cols, m, n = [1, 1], [0, 2, 0], 2, 3
odd_r = sum(1 for x in rows if x % 2 == 1)
odd_c = sum(1 for x in cols if x % 2 == 1)
print("odd_r =", odd_r, "odd_c =", odd_c)                     # 2 0
print(odd_r * (n - odd_c) + (m - odd_r) * odd_c)              # 2*3 + 0*0 = 6

odd_r = 2 odd_c = 0
6


In [13]:
def odd_cells_formula(m: int, n: int, indices: list[list[int]]) -> int:
    rows = [0] * m
    cols = [0] * n
    for r, c in indices:
        rows[r] += 1
        cols[c] += 1
    odd_r = sum(1 for x in rows if x % 2 == 1)
    odd_c = sum(1 for x in cols if x % 2 == 1)
    return odd_r * (n - odd_c) + (m - odd_r) * odd_c

print(odd_cells_formula(2, 3, [[0, 1], [1, 1]]))   # 6
print(odd_cells_formula(2, 2, [[1, 1], [0, 0]]))   # 0

6
0


## 8. Cách 4 — Chỉ lưu bit chẵn/lẻ bằng XOR (`^= 1`) hoặc `not`

Ta không cần **đếm chính xác** số lần, chỉ cần biết **chẵn hay lẻ**. Mỗi lần hàng `r` được chọn, parity của nó **đảo**:
0 → 1 → 0 → 1 … Phép `x ^= 1` làm đúng việc đảo đó (`0 ^ 1 = 1`, `1 ^ 1 = 0`).

Ngoài ra Python cho phép cộng các `bool`: `True` được tính là `1`, nên `sum(rows)` đếm luôn số hàng lẻ.

In [14]:
p = 0
for step in range(5):
    p ^= 1
    print(f"sau lần đảo {step + 1}: p = {p}")
print(True + True + False)    # 2
print(sum([True, False, True, True]))   # 3

sau lần đảo 1: p = 1
sau lần đảo 2: p = 0
sau lần đảo 3: p = 1
sau lần đảo 4: p = 0
sau lần đảo 5: p = 1
2
3


In [15]:
def odd_cells_xor(m: int, n: int, indices: list[list[int]]) -> int:
    rows = [False] * m
    cols = [False] * n
    for r, c in indices:
        rows[r] = not rows[r]      # đảo parity hàng r
        cols[c] = not cols[c]      # đảo parity cột c
    odd_r = sum(rows)              # True = 1
    odd_c = sum(cols)
    return odd_r * (n - odd_c) + (m - odd_r) * odd_c

print(odd_cells_xor(2, 3, [[0, 1], [1, 1]]))   # 6
print(odd_cells_xor(2, 2, [[1, 1], [0, 0]]))   # 0

6
0


## 9. Trace từng bước (Cách 3)

Ví dụ khó hơn để thấy cả hàng lẻ lẫn cột lẻ: `m = 3, n = 4, indices = [[0,0],[1,2],[0,3]]`.

| bước | `[r, c]` | `rows` sau | `cols` sau |
|---:|---|---|---|
| 1 | `[0,0]` | `[1,0,0]` | `[1,0,0,0]` |
| 2 | `[1,2]` | `[1,1,0]` | `[1,0,1,0]` |
| 3 | `[0,3]` | `[2,1,0]` | `[1,0,1,1]` |

- Hàng lẻ: chỉ hàng 1 → `odd_r = 1`; hàng chẵn: `3 - 1 = 2`.
- Cột lẻ: cột 0, 2, 3 → `odd_c = 3`; cột chẵn: `4 - 3 = 1`.
- Đáp án: `1 * 1 + 2 * 3 = 7`.

In [16]:
def trace_odd_cells(m, n, indices):
    rows = [0] * m
    cols = [0] * n
    for step, (r, c) in enumerate(indices, start=1):
        rows[r] += 1
        cols[c] += 1
        print(f"bước {step}: [r,c]=[{r},{c}] -> rows={rows}, cols={cols}")
    odd_r = sum(1 for x in rows if x % 2)
    odd_c = sum(1 for x in cols if x % 2)
    print(f"odd_r={odd_r} (chẵn {m - odd_r}), odd_c={odd_c} (chẵn {n - odd_c})")
    ans = odd_r * (n - odd_c) + (m - odd_r) * odd_c
    print(f"answer = {odd_r}*{n - odd_c} + {m - odd_r}*{odd_c} = {ans}")
    # đối chiếu bằng ma trận thật
    grid = [[rows[i] + cols[j] for j in range(n)] for i in range(m)]
    for row in grid:
        print("   ", row, "".join("L" if v % 2 else "." for v in row))
    return ans

trace_odd_cells(3, 4, [[0, 0], [1, 2], [0, 3]])

bước 1: [r,c]=[0,0] -> rows=[1, 0, 0], cols=[1, 0, 0, 0]
bước 2: [r,c]=[1,2] -> rows=[1, 1, 0], cols=[1, 0, 1, 0]
bước 3: [r,c]=[0,3] -> rows=[2, 1, 0], cols=[1, 0, 1, 1]
odd_r=1 (chẵn 2), odd_c=3 (chẵn 1)
answer = 1*1 + 2*3 = 7
    [3, 2, 3, 3] L.LL
    [2, 1, 2, 2] .L..
    [1, 0, 1, 1] L.LL


7

### Giải thích `enumerate(indices, start=1)` và `(r, c)`

- `enumerate(x)` cho từng cặp `(chỉ_số, phần_tử)`; `start=1` để đếm bước từ 1 cho dễ đọc.
- Phần tử ở đây là `[r, c]`, nên ta viết `step, (r, c)` để tách 2 tầng cùng lúc.

Trong bảng in ra, `L` là ô lẻ: hàng 1 (hàng lẻ) có `L` ở cột chẵn (cột 1); hàng 0 và 2 (hàng chẵn) có `L` ở 3 cột lẻ → tổng `1 + 3 + 3 = 7`.

In [17]:
for step, (r, c) in enumerate([[0, 0], [1, 2]], start=1):
    print(step, r, c)

1 0 0
2 1 2


## 10. Code nộp LeetCode (lời giải chính — Cách 3)

In [18]:
class Solution:
    def oddCells(self, m: int, n: int, indices: list[list[int]]) -> int:
        rows = [0] * m
        cols = [0] * n
        for r, c in indices:
            rows[r] ^= 1          # chỉ giữ parity
            cols[c] ^= 1
        odd_r = sum(rows)
        odd_c = sum(cols)
        return odd_r * (n - odd_c) + (m - odd_r) * odd_c

sol = Solution()
print(sol.oddCells(2, 3, [[0, 1], [1, 1]]))   # 6
print(sol.oddCells(2, 2, [[1, 1], [0, 0]]))   # 0

6
0


## 11. Vì sao đúng? (invariant)

1. **Tính độc lập:** mỗi thao tác `[r, c]` cộng 1 vào ô `(i, j)` đúng một lần nếu `i == r`, đúng một lần nếu `j == c` (hai lần nếu cả hai). Nên sau mọi thao tác, `grid[i][j] = rows[i] + cols[j]` — thứ tự thao tác **không quan trọng** (phép cộng giao hoán).
2. **Invariant của vòng lặp:** sau khi xử lý `k` phần tử đầu, `rows[i]` (bản `^=1`) bằng parity số lần hàng `i` xuất hiện trong `k` phần tử đó; tương tự với `cols`.
3. **Đếm không trùng, không sót:** ô `(i, j)` lẻ ⇔ đúng một trong `rows[i]`, `cols[j]` lẻ. Hai loại "hàng lẻ–cột chẵn" và "hàng chẵn–cột lẻ" không giao nhau và phủ hết các ô lẻ; nguyên lý nhân cho số ô từng loại.

## 12. Độ phức tạp và so sánh

Gọi `L = len(indices)`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ |
|---|---|---|---|
| 1. Mô phỏng | cập nhật cả hàng/cột trên ma trận | `O(L·(m+n) + m·n)` | `O(m·n)` |
| 2. Đếm hàng/cột | `grid[r][c] = rows[r] + cols[c]`, duyệt mọi ô | `O(L + m·n)` | `O(m + n)` |
| 3. Công thức | `odd_r·(n−odd_c) + (m−odd_r)·odd_c` | `O(L + m + n)` | `O(m + n)` |
| 4. Parity bool/XOR | như 3 nhưng chỉ lưu bit | `O(L + m + n)` | `O(m + n)` |

Cách 3/4 đáp ứng đúng follow-up.

## 13. Bộ test

Nhóm test:
1. Ví dụ trong đề.
2. Biên: ma trận `1 x 1`; một thao tác; thao tác lặp lại cùng vị trí.
3. Phản ví dụ cho công thức "sai" thường gặp (xem mục 14).
4. Đối chiếu 4 cách trên nhiều input ngẫu nhiên (dùng `random` với `seed` cố định để kết quả lặp lại được).

In [19]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.oddCells(2, 3, [[0, 1], [1, 1]]), 6)
check("đề 2", sol.oddCells(2, 2, [[1, 1], [0, 0]]), 0)
check("1x1 một lần", sol.oddCells(1, 1, [[0, 0]]), 0)          # ô bị cộng 2 lần
check("1x3 một lần", sol.oddCells(1, 3, [[0, 1]]), 2)          # [1,2,1]
check("lặp cùng vị trí 2 lần", sol.oddCells(2, 2, [[0, 0], [0, 0]]), 0)
check("lặp 3 lần", sol.oddCells(2, 2, [[0, 0]] * 3), 2)
check("trace 3x4", sol.oddCells(3, 4, [[0, 0], [1, 2], [0, 3]]), 7)

PASS đề 1: 6
PASS đề 2: 0
PASS 1x1 một lần: 0
PASS 1x3 một lần: 2
PASS lặp cùng vị trí 2 lần: 0
PASS lặp 3 lần: 2
PASS trace 3x4: 7


In [20]:
import random
random.seed(1252)
methods = [odd_cells_simulate, odd_cells_count_rc, odd_cells_formula, odd_cells_xor]
for _ in range(300):
    m = random.randint(1, 8)
    n = random.randint(1, 8)
    L = random.randint(1, 15)
    idx = [[random.randrange(m), random.randrange(n)] for _ in range(L)]
    expected = odd_cells_simulate(m, n, idx)
    for f in methods:
        assert f(m, n, idx) == expected, (f.__name__, m, n, idx)
    assert Solution().oddCells(m, n, idx) == expected
print("300 bộ ngẫu nhiên: 4 cách + Solution đều khớp")

300 bộ ngẫu nhiên: 4 cách + Solution đều khớp


## 14. Lỗi người mới hay gặp

### Lỗi 1: Tạo ma trận bằng `[[0] * n] * m`

Như mục 2.2: các hàng chung một object → cộng hàng 0 cũng làm "cộng" hàng 1.

In [21]:
def wrong_alias(m, n, indices):
    grid = [[0] * n] * m                   # SAI
    for r, c in indices:
        for col in range(n):
            grid[r][col] += 1
        for row in range(m):
            grid[row][c] += 1
    return sum(1 for row in grid for v in row if v % 2 == 1)

print("sai:", wrong_alias(2, 3, [[0, 1], [1, 1]]), "| đúng:", odd_cells_simulate(2, 3, [[0, 1], [1, 1]]))

sai: 0 | đúng: 6


Kết quả sai vì mỗi lệnh `grid[row][c] += 1` trong vòng cột thực ra cộng `m` lần vào **cùng một** list. Sửa: `[[0] * n for _ in range(m)]`.

### Lỗi 2: Nghĩ ô giao chỉ được cộng 1 lần

Có bạn "tránh cộng trùng" ô `(r, c)` khi cập nhật cột. Nhưng đề nói làm **cả hai** việc → ô giao được cộng 2. Ví dụ `1x1, [[0,0]]` phải ra 2 (chẵn) → đáp án 0.

In [22]:
def wrong_skip_intersection(m, n, indices):
    grid = [[0] * n for _ in range(m)]
    for r, c in indices:
        for col in range(n):
            grid[r][col] += 1
        for row in range(m):
            if row != r:                    # SAI: bỏ ô giao
                grid[row][c] += 1
    return sum(1 for row in grid for v in row if v % 2 == 1)

print("sai:", wrong_skip_intersection(1, 1, [[0, 0]]), "| đúng:", odd_cells_simulate(1, 1, [[0, 0]]))

sai: 1 | đúng: 0


### Lỗi 3: Công thức "số hàng lẻ × số cột lẻ"

Đếm ô mà **cả hai** lẻ là sai: lẻ + lẻ = chẵn! Phải đếm ô có **đúng một** thành phần lẻ.

In [23]:
def wrong_formula(m, n, indices):
    rows = [0] * m; cols = [0] * n
    for r, c in indices:
        rows[r] ^= 1; cols[c] ^= 1
    return sum(rows) * sum(cols)           # SAI

print("sai:", wrong_formula(2, 3, [[0, 1], [1, 1]]), "| đúng:", odd_cells_formula(2, 3, [[0, 1], [1, 1]]))

sai: 0 | đúng: 6


### Lỗi 4: Nhầm `m` (số hàng) với `n` (số cột)

`rows` phải dài `m`, `cols` dài `n`. Nếu đảo, ma trận không vuông sẽ bị `IndexError` hoặc ra số sai. Mẹo nhớ: `grid[r][c]` — **r**ow trước, **c**olumn sau, giống thứ tự `m x n`.

### Lỗi 5: Dùng `rows[r] += 1` nhưng quên `% 2` khi đếm

Nếu lưu số lần thật (không `^=1`) thì `sum(rows)` **không** phải số hàng lẻ. Phải `sum(1 for x in rows if x % 2)`.

In [24]:
rows = [2, 1, 0]
print("sum(rows) =", sum(rows), "(SAI ý nghĩa)")
print("số hàng lẻ =", sum(1 for x in rows if x % 2))   # 1

sum(rows) = 3 (SAI ý nghĩa)
số hàng lẻ = 1


## 15. Mẹo nhận dạng pattern và biến thể

**Dấu hiệu nhận dạng:**
- Thao tác "cộng cả hàng / cả cột / cả đoạn" lặp lại nhiều lần → thay vì cập nhật từng ô, **ghi nhận thao tác theo chiều** (hàng, cột) rồi ghép lại ở cuối.
- Đề chỉ hỏi chẵn/lẻ → chỉ cần lưu **parity** (1 bit), dùng `^= 1`.
- Câu hỏi "đếm số ô thỏa điều kiện kết hợp hàng–cột" → thường đưa về **nguyên lý nhân**: (số hàng loại A) × (số cột loại B).

**Biến thể / bài liên quan:**
- LeetCode 598 *Range Addition II*: mỗi thao tác cộng 1 vào hình chữ nhật góc trái trên → đáp án là `min(a) * min(b)`.
- LeetCode 370 *Range Addition* (Premium): nhiều thao tác cộng cả đoạn → ghi nhận thao tác rồi gộp ở cuối (mảng hiệu), cùng tinh thần "đừng cập nhật từng ô".
- Biến thể tự chế: đếm ô **chẵn** (= `m·n − số ô lẻ`); hoặc trả về chính ma trận cuối bằng `rows[r] + cols[c]`.

In [25]:
# Mini demo biến thể: số ô chẵn = tổng ô - số ô lẻ
m, n, idx = 2, 3, [[0, 1], [1, 1]]
odd = odd_cells_formula(m, n, idx)
print("ô lẻ:", odd, "| ô chẵn:", m * n - odd)   # 6 | 0

# Range Addition II (598) mini: ops = [[2,2],[3,3]] trên ma trận 3x3
ops = [[2, 2], [3, 3]]
print("598 ->", min(a for a, _ in ops) * min(b for _, b in ops))   # 4

ô lẻ: 6 | ô chẵn: 0
598 -> 4


## 16. Bài tập tự luyện (tăng dần)

> Tự làm trước. Gợi ý ở mục 17, đáp án ở mục 18 (cuối notebook). Các cell dưới chỉ định nghĩa hàm, **không** gọi.

### Bài 1 — Tính tay
Với `m = 2, n = 2, indices = [[0,0],[0,1]]`, viết ma trận cuối và số ô lẻ. (Ghi đáp án vào biến `ex1_answer`.)

In [26]:
ex1_answer = None   # TODO: thay None bằng số ô lẻ bạn tính được

# Kiểm tra sau khi làm:
# assert ex1_answer == odd_cells_simulate(2, 2, [[0, 0], [0, 1]])

### Bài 2 — Dựng lại ma trận cuối
Viết `build_final_grid(m, n, indices)` trả về ma trận cuối **không** mô phỏng từng ô, mà dùng `rows[r] + cols[c]`.

In [27]:
def build_final_grid(m: int, n: int, indices: list[list[int]]) -> list[list[int]]:
    # TODO 1: tạo rows, cols
    # TODO 2: đếm
    # TODO 3: dựng ma trận bằng list comprehension
    pass

# assert build_final_grid(2, 3, [[0, 1], [1, 1]]) == [[1, 3, 1], [1, 3, 1]]

### Bài 3 — Đếm ô chẵn bằng công thức
Viết `even_cells(m, n, indices)` trong `O(L + m + n)`, không dùng `m*n - odd`; hãy suy ra công thức trực tiếp (hàng chẵn×cột chẵn + hàng lẻ×cột lẻ).

In [28]:
def even_cells(m: int, n: int, indices: list[list[int]]) -> int:
    # TODO
    pass

# assert even_cells(2, 2, [[1, 1], [0, 0]]) == 4
# assert even_cells(2, 3, [[0, 1], [1, 1]]) == 0

### Bài 4 — Cộng `k` thay vì cộng 1
Mỗi phần tử là `[r, c, k]`: cộng `k` vào cả hàng `r` và cả cột `c`. Đếm ô lẻ trong `O(L + m + n)`.

In [29]:
def odd_cells_weighted(m: int, n: int, ops: list[list[int]]) -> int:
    # TODO: parity của tổng chỉ phụ thuộc parity của k
    pass

# assert odd_cells_weighted(2, 3, [[0, 1, 1], [1, 1, 1]]) == 6
# assert odd_cells_weighted(2, 3, [[0, 1, 2]]) == 0
# assert odd_cells_weighted(1, 3, [[0, 1, 3]]) == 2

### Bài 5 — Mini challenge: chỉ cộng hàng HOẶC cột
Mỗi phần tử là `("R", i)` (cộng 1 cả hàng `i`) hoặc `("C", j)` (cộng 1 cả cột `j`). Đếm ô lẻ.

In [30]:
def odd_cells_mixed(m: int, n: int, ops: list[tuple[str, int]]) -> int:
    # TODO
    pass

# assert odd_cells_mixed(2, 3, [("R", 0)]) == 3
# assert odd_cells_mixed(2, 3, [("R", 0), ("C", 1)]) == 3
# assert odd_cells_mixed(2, 2, [("R", 0), ("R", 0)]) == 0

## 17. Gợi ý 3 mức

**Bài 1:** (1) Làm như mục 1. (2) `rows = [2, 0]`, `cols = [1, 1]`. (3) `grid[r][c] = rows[r] + cols[c]`, đếm lẻ.

**Bài 2:** (1) Insight mục 5. (2) Cần 2 list đếm dài `m` và `n`. (3) `[[rows[r] + cols[c] for c in range(n)] for r in range(m)]`.

**Bài 3:** (1) Tổng chẵn khi hai số cùng parity. (2) Có 2 nhóm: chẵn–chẵn và lẻ–lẻ. (3) `(m - odd_r) * (n - odd_c) + odd_r * odd_c`.

**Bài 4:** (1) Cộng `k` chẵn không đổi parity. (2) Chỉ đảo khi `k % 2 == 1`. (3) `rows[r] ^= k & 1; cols[c] ^= k & 1`.

**Bài 5:** (1) Vẫn là `rows[i] + cols[j]`. (2) Thao tác "R" chỉ chạm `rows`, "C" chỉ chạm `cols`. (3) Dùng lại công thức mục 7.

## 18. Đáp án tham khảo

Chỉ xem sau khi đã tự làm. Cell dưới định nghĩa đáp án và **chạy assert** để chứng minh đúng.

In [31]:
print("ma trận bài 1:", [[[2, 0][r] + [1, 1][c] for c in range(2)] for r in range(2)])
ans_ex1 = odd_cells_formula(2, 2, [[0, 0], [0, 1]])
print("số ô lẻ bài 1:", ans_ex1)


def ans_build_final_grid(m, n, indices):
    rows = [0] * m
    cols = [0] * n
    for r, c in indices:
        rows[r] += 1
        cols[c] += 1
    return [[rows[r] + cols[c] for c in range(n)] for r in range(m)]


def ans_even_cells(m, n, indices):
    rows = [0] * m
    cols = [0] * n
    for r, c in indices:
        rows[r] ^= 1
        cols[c] ^= 1
    odd_r, odd_c = sum(rows), sum(cols)
    return (m - odd_r) * (n - odd_c) + odd_r * odd_c


def ans_odd_cells_weighted(m, n, ops):
    rows = [0] * m
    cols = [0] * n
    for r, c, k in ops:
        rows[r] ^= k & 1
        cols[c] ^= k & 1
    odd_r, odd_c = sum(rows), sum(cols)
    return odd_r * (n - odd_c) + (m - odd_r) * odd_c


def ans_odd_cells_mixed(m, n, ops):
    rows = [0] * m
    cols = [0] * n
    for kind, i in ops:
        if kind == "R":
            rows[i] ^= 1
        else:
            cols[i] ^= 1
    odd_r, odd_c = sum(rows), sum(cols)
    return odd_r * (n - odd_c) + (m - odd_r) * odd_c


assert ans_ex1 == odd_cells_simulate(2, 2, [[0, 0], [0, 1]]) == 4
assert ans_build_final_grid(2, 3, [[0, 1], [1, 1]]) == [[1, 3, 1], [1, 3, 1]]
assert ans_even_cells(2, 2, [[1, 1], [0, 0]]) == 4
assert ans_even_cells(2, 3, [[0, 1], [1, 1]]) == 0
assert ans_odd_cells_weighted(2, 3, [[0, 1, 1], [1, 1, 1]]) == 6
assert ans_odd_cells_weighted(2, 3, [[0, 1, 2]]) == 0
assert ans_odd_cells_weighted(1, 3, [[0, 1, 3]]) == 2
assert ans_odd_cells_mixed(2, 3, [("R", 0)]) == 3
assert ans_odd_cells_mixed(2, 3, [("R", 0), ("C", 1)]) == 3
assert ans_odd_cells_mixed(2, 2, [("R", 0), ("R", 0)]) == 0

import random
random.seed(7)
for _ in range(200):
    m, n = random.randint(1, 6), random.randint(1, 6)
    idx = [[random.randrange(m), random.randrange(n)] for _ in range(random.randint(1, 10))]
    assert ans_even_cells(m, n, idx) == m * n - odd_cells_simulate(m, n, idx)
print("Đáp án bài tập: tất cả assert PASS")

ma trận bài 1: [[3, 3], [1, 1]]
số ô lẻ bài 1: 4
Đáp án bài tập: tất cả assert PASS


Ghi chú bài 1: `rows = [2, 0]`, `cols = [1, 1]` → ma trận `[[3, 3], [1, 1]]` → cả 4 ô lẻ, đáp án `4`.

## 19. Template nhớ nhanh + checklist submit

```python
class Solution:
    def oddCells(self, m, n, indices):
        rows, cols = [0] * m, [0] * n
        for r, c in indices:
            rows[r] ^= 1
            cols[c] ^= 1
        odd_r, odd_c = sum(rows), sum(cols)
        return odd_r * (n - odd_c) + (m - odd_r) * odd_c
```

**Câu thần chú:** *"ô = hàng + cột; lẻ ⇔ đúng một bên lẻ; đếm bằng nhân."*

**Checklist trước khi submit:**
- [ ] `rows` dài `m`, `cols` dài `n` (không đảo).
- [ ] Ô giao được cộng **2 lần** — không né.
- [ ] Đếm ô có **đúng một** thành phần lẻ (không phải cả hai).
- [ ] Nếu tạo ma trận: dùng `[[0] * n for _ in range(m)]`, không dùng `[[0] * n] * m`.
- [ ] Đã test ví dụ đề + `1x1` + thao tác lặp lại.